# Nova Workshop 2026
<br/>
<img src="https://www.polyestertime.com/wp-content/uploads/2017/01/Nova-Chemical-23-09-2016.jpg" />
<br/><br/>

## Introduction
Having completed our application back-end, we will now:

1. Build a `build_incident_context(line_id)` function that returns recent KPIs and incidents.
2. Call an external model/agent endpoint `nova-incident-analyst` (Model Serving) with that context and a question.
3. Display the structured response.

Goal:
- Prototype the logic for the "Incident Copilot" before wiring it into the Databricks App.


In [0]:
import pyspark.sql.functions as F

user_schema = "andrij_demo"          # TODO
catalog_name = "nova_workshop"
lakebase_db = "nova_incidents"

gold_table = f"{catalog_name}.{user_schema}.gold_daily_line_kpis"
incidents_table = f"{catalog_name}.{user_schema}.gold_incidents"

print("Gold KPIs table :", gold_table)
print("Incidents table :", incidents_table)

## Step 1 – Build Incident Context

We retrieve:
- Last 14 days of KPIs for the line.
- Last 20 incidents for the line.

The function will return a Python dict suitable for JSON serialization.

In [0]:
def build_incident_context(line_id: str, days: int = 14, max_incidents: int = 20):
    kpis_df = (
        spark.read.table(gold_table)
             .filter(F.col("line_id") == line_id)
             .orderBy(F.col("day").desc())
             .limit(days)
    )

    incidents_df = (
        spark.read.table(incidents_table)
             .filter(F.col("line_id") == line_id)
             .orderBy(F.col("ts").desc())
             .limit(max_incidents)
    )

    kpis = kpis_df.toPandas().to_dict(orient="records")
    incidents = incidents_df.toPandas().to_dict(orient="records")

    context = {
        "recent_kpis": kpis,
        "recent_incidents": incidents,
    }
    return context

# Quick sanity check
ctx = build_incident_context("LINE_01")
ctx.keys(), len(ctx["recent_kpis"]), len(ctx["recent_incidents"])

## Genie Space Setup Instructions

Follow these steps to create a Genie Space with your relevant tables and add it as an agent in AgentBricks:

### 1. Create Genie Space

- Go to the Genie Space UI in Databricks.
- Click **Create Space** and name it (e.g., `Nova Workshop Supervisor`).
- Add the following tables to your Genie Space:
  - `nova_workshop.andrij_demo.gold_daily_line_kpis`
  - `nova_workshop.andrij_demo.incidents` *(use this instead of `gold_incidents` for live incident data)*

### 2. Add Genie Space as Agent in AgentBricks

- In AgentBricks, navigate to **Agents** and click **Add Agent**.
- Select your Genie Space (`Nova Workshop Supervisor`) as the agent source.
- Assign the agent the **Supervisor** role for context-aware access.
- Enable **Model Serving Endpoint** capabilities:
  - Specify the endpoint (e.g., `nova-incident-analyst`) and configure authentication (token, URL).
- Save the agent configuration.

Your Genie Space agent is now ready to provide context and interact with the model serving endpoint as a supervisor.

## Step 2 – Call Model/Agent Endpoint

Assumptions:
- A Model Serving endpoint (e.g. `nova-incident-analyst`) is deployed.
- URL and token are available via environment variables:
  - `INCIDENT_ANALYST_URL`
  - `INCIDENT_ANALYST_TOKEN`

We'll call the endpoint with JSON:
```json
{
  "question": "...",
  "context": {
    "recent_kpis": [...],
    "recent_incidents": [...]
  }
}


In [0]:
import os
import requests
import json

#These variables will need to be pre-loaded before the workshop
# INCIDENT_ANALYST_URL = os.getenv("INCIDENT_ANALYST_URL")
# INCIDENT_ANALYST_TOKEN = os.getenv("INCIDENT_ANALYST_TOKEN")
INCIDENT_ANALYST_URL = "https://dbc-9c7dbe12-0a2f.cloud.databricks.com/serving-endpoints/mas-0f8edf80-endpoint/invocations"
INCIDENT_ANALYST_TOKEN = "dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()"

# How to get your Databricks token: https://docs.databricks.com/en/dev-tools/auth/pat.html
#DATABRICKS_TOKEN = os.environ.get('DATABRICKS_TOKEN')
# Alternatively in a Databricks notebook you can use this:
# DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()


print("INCIDENT_ANALYST_URL:", INCIDENT_ANALYST_URL)

In [0]:
def call_incident_analyst(question: str, line_id: str):
    if not INCIDENT_ANALYST_URL or not INCIDENT_ANALYST_TOKEN:
        raise RuntimeError("INCIDENT_ANALYST_URL or INCIDENT_ANALYST_TOKEN is not set.")

    context = build_incident_context(line_id)
    payload = {
        "question": question,
        "context": context,
    }

    headers = {
        "Authorization": f"Bearer {INCIDENT_ANALYST_TOKEN}",
        "Content-Type": "application/json",
    }

    resp = requests.post(INCIDENT_ANALYST_URL, headers=headers, json=payload)
    resp.raise_for_status()
    return resp.json()

## Step 3 – Test the Copilot

Ask a question about your line, using real context from your Gold and incidents tables.

In [0]:
# The build_incident_context function retrieves recent KPIs and incidents for a given production line.
# It is used to build a context dictionary for downstream analysis or model inference.

# References:
# - gold_table and incidents_table are defined in Cell 2.
# - The function is described in Cell 3 and used in Cell 4 and Cell 10.

def build_incident_context(line_id: str, days: int = 14, max_incidents: int = 20):
    # Retrieve last 'days' rows of KPIs for the specified line_id, ordered by day descending
    kpis_df = (
        spark.read.table(gold_table)
             .filter(F.col("line_id") == line_id)
             .orderBy(F.col("day").desc())
             .limit(days)
    )

    # Retrieve last 'max_incidents' rows of incidents for the specified line_id, ordered by timestamp descending
    incidents_df = (
        spark.read.table(incidents_table)
             .filter(F.col("line_id") == line_id)
             .orderBy(F.col("ts").desc())
             .limit(max_incidents)
    )

    # Convert datetime columns to string for JSON serialization in KPIs
    kpis_pdf = kpis_df.toPandas()
    for col in kpis_pdf.select_dtypes(include=["datetime", "datetimetz"]).columns:
        kpis_pdf[col] = kpis_pdf[col].astype(str)
    kpis = kpis_pdf.to_dict(orient="records")

    # Convert datetime columns to string for JSON serialization in incidents
    incidents_pdf = incidents_df.toPandas()
    for col in incidents_pdf.select_dtypes(include=["datetime", "datetimetz"]).columns:
        incidents_pdf[col] = incidents_pdf[col].astype(str)
    incidents = incidents_pdf.to_dict(orient="records")

    # Return context dictionary containing recent KPIs and incidents
    context = {
        "recent_kpis": kpis,
        "recent_incidents": incidents,
    }
    return context

In [0]:
# This code demonstrates how to call a Databricks Model Serving endpoint using the OpenAI client.
# It builds a context for a specific production line, sends a question and context to the endpoint,
# and prints the model's response.

# 1. The OpenAI client is initialized with a Databricks personal access token (DATABRICKS_TOKEN)
#    and the base URL for the Databricks Model Serving endpoint.
# 2. The context for the specified line is built using build_incident_context.
# 3. The model is invoked with a user question and the context.
# 4. The response is printed by extracting the text from the model's output.

# Suggestions for Databricks App integration:
# - DATABRICKS_TOKEN: In a Databricks App, you should not hardcode or use notebook context to fetch the token.
#   Instead, securely pass the user's token from the app frontend to the backend, or use OAuth on behalf of the user.
#   The backend should use the token associated with the current user/session for authentication.
# - Endpoint URL: Parameterize the endpoint URL so it can be configured per environment (dev, staging, prod).
# - Error Handling: Add robust error handling for failed requests and invalid responses.
# - Security: Never expose tokens in client-side code or logs.

client = OpenAI(
    api_key=DATABRICKS_TOKEN,  # In a Databricks App, obtain this securely per user/session.
    base_url="https://dbc-9c7dbe12-0a2f.cloud.databricks.com/serving-endpoints"
)

test_line_id = "LINE_01"
question = "Why is BAD rate increasing on my line, and what checks should I run next?"
context = build_incident_context(test_line_id)

response = client.responses.create(
    #IMPORTANT: Model will need to be updated
    model="mas-0f8edf80-endpoint",
    input=[
        {
            "role": "user",
            "content": question,
            "context": context
        }
    ]
)

print(" ".join(getattr(content, "text", "") for output in response.output for content in getattr(output, "content", [])))